# Notebook 02: Hệ thống câu hỏi nghiên cứu và Tổng quan phân tích chuyên sâu (Master Hub)
**Đề tài:** Phân tích sự cố bất lợi thuốc thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Đối chiếu xu hướng 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm thực hiện:** Global Ban

> **Phân rã notebook 02 theo câu hỏi:**  
> Để tối ưu hóa quy trình làm việc nhóm, chống xung đột phiên bản Git và đảm bảo tính độc lập chuyên sâu, toàn bộ phân tích chi tiết cho 5 câu hỏi Non-ML được tổ chức thành **5 Notebook riêng biệt**:
> 1. **[`02_q1_analysis.ipynb`](02_q1_analysis.ipynb)**: **Câu hỏi Q1** — Phản ứng VeDDRA & Xu hướng theo quý
> 2. **[`02_q2_analysis.ipynb`](02_q2_analysis.ipynb)**: **Câu hỏi Q2** — Đặc điểm sinh học & Biến cố nghiêm trọng
> 3. **[`02_q3_analysis.ipynb`](02_q3_analysis.ipynb)**: **Câu hỏi Q3** — Tỷ suất đồng xuất hiện ROR Hoạt chất x Triệu chứng
> 4. **[`02_q4_analysis.ipynb`](02_q4_analysis.ipynb)**: **Câu hỏi Q4** — Dùng thuốc sai nhãn Off-label & Kết cục y khoa
> 5. **[`02_q5_analysis.ipynb`](02_q5_analysis.ipynb)**: **Câu hỏi Q5** — Dịch tễ đàn & Tỷ lệ ảnh hưởng ở đàn
>
> File Master này (`02_analysis.ipynb`) đóng vai trò là **Trung tâm Điều phối**: Trình bày Mục 3.3 (Hệ thống câu hỏi và Ma trận độ sâu D1–D6), cung cấp liên kết trực tiếp sang từng câu hỏi Q1–Q5 và tổng hợp Kết quả then chốt của 5 câu hỏi phân tích.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.visualization import set_project_plot_style
set_project_plot_style()

# Tải 4 bảng dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
drugs = data['drugs']
reactions = data['reactions']
outcomes = data['outcomes']
print(f"[Data Loaded] Reports: {reports.shape}, Drugs: {drugs.shape}, Reactions: {reactions.shape}, Outcomes: {outcomes.shape}")

[Data Loaded] Reports: (94412, 33), Drugs: (174123, 31), Reactions: (270244, 8), Outcomes: (85143, 5)


---
## 1. Tổng quan hệ thống câu hỏi nghiên cứu và độ sâu


| Mã | Chủ sở hữu | Câu hỏi nghiên cứu | Tiêu chí độ sâu đáp ứng | Notebook thực thi |
|---|---|---|---|---|
| **Q1** | **M1 — Khang** | Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài khác trong năm 2025, và xu hướng biến động theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của chúng diễn biến ra sao khi bước sang nửa đầu năm 2026 (Q1)? | **D1** (Combining tables), **D2** (Reshaping & time series), **D5** (Quantifying uncertainty) | [`02_q1_analysis.ipynb`](02_q1_analysis.ipynb) |
| **Q2** | **M2 — Nhi** | Những đặc điểm sinh học nào (giai đoạn sống, trọng lượng cơ thể, giới tính) có liên quan đến các biến cố bất lợi nghiêm trọng, sau khi đã kiểm soát yếu tố loài và số lượng thuốc sử dụng? | **D3** (Derived variables), **D4** (Controlling for confounder), **D5** (CI/Hypothesis test), **D6** (Robustness check) | [`02_q2_analysis.ipynb`](02_q2_analysis.ipynb) |
| **Q3** | **M3 — Kiệt** | Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào có tỷ suất đồng xuất hiện (Reporting Odds Ratio) đặc trưng nhất với từng nhóm phản ứng cụ thể, và cấu trúc này khác biệt ra sao giữa **Safety Issue** và **Lack Of Expected Effectiveness** trong năm 2025? | **D1** (Multi-table merge), **D2** (Pivot/melt matrix), **D4** (Subgroup control), **D5** (Odds Ratio & 95% CI) | [`02_q3_analysis.ipynb`](02_q3_analysis.ipynb) |
| **Q4** | **M3 — Kiệt** | Việc dùng thuốc sai nhãn (`off-label`) kết hợp với đa trị liệu (`polypharmacy`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục (`outcome`) sau khi kiểm soát biến nhiễu là **loài** và **đối tượng báo cáo (`primary_reporter`)**? | **D1** (Combining tables), **D3** (Derived indicators), **D4** (Controlling for confounder), **D6** (Robustness check) | [`02_q4_analysis.ipynb`](02_q4_analysis.ipynb) |
| **Q5** | **M2 — Nhi** | Những yếu tố nào có liên quan đến tỷ lệ động vật bị ảnh hưởng trong tổng số động vật được điều trị ở mỗi đàn? | **D1** (Multi-table join for herd data), **D3** (Derived attack rate), **D4** (Subgroup / species control), **D5** (CI & regression), **D6** (Robustness: exclude attack_rate > 1 vs. keep) | [`02_q5_analysis.ipynb`](02_q5_analysis.ipynb) |
| **Q6 (ML)** | **M1 — Khang** | Huấn luyện mô hình trên dữ liệu 2025 (Q1–Q4) để dự đoán khả năng một báo cáo trong năm 2026 (Q1) là sự cố nghiêm trọng (`serious_ae`) dựa trên thông tin ban đầu lúc tiếp nhận. | **ML Checklist 1–6** (Time-based split, Pipeline, Leakage-free, Tuning, Error Analysis, Permutation Importance) | [`03_ml_question.ipynb`](03_ml_question.ipynb) |


---
## 2. Câu hỏi Q1: Phản ứng VeDDRA hàng đầu & Xu hướng Biến động theo Quý
- **Chủ sở hữu:** Đinh Công Khang
- **Notebook chi tiết:** [`02_q1_analysis.ipynb`](02_q1_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Top phản ứng VeDDRA theo loài:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb` khi hoàn tất phân tích)
  - *Dấu hiệu mùa vụ & Biến động 2025 $\rightarrow$ 2026 Q1:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb`)
  - *Giá trị p-value kiểm định $\chi^2$ & Bootstrap 95% CI:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb`)


---
## 3. Câu hỏi Q2: Đặc điểm Sinh học & Nguy cơ Biến cố Nghiêm trọng
- **Chủ sở hữu:** Trần Ngô Uyên Nhi
- **Notebook chi tiết:** [`02_q2_analysis.ipynb`](02_q2_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Ảnh hưởng của giai đoạn sống & Cân nặng (Baseline năm 2025):* Trên `58,256` ca hoàn chỉnh năm 2025 (`47,858` Chó, `10,398` Mèo), thú già (`Senior`) tăng nguy cơ ở cả hai loài, còn cân nặng có tác động **đảo chiều**: Chó nặng cân có nguy cơ cao hơn (`Q1` 42.07% → `Q4` 47.73%), ngược lại Mèo nhẹ cân có nguy cơ cao hơn (`Q1` 39.34% → `Q4` 31.00%). Đặc biệt, đường cong chữ U ở Mèo non (`Junior`) biến mất khi kiểm soát cân nặng và đa trị liệu.
  - *Tỷ suất chênh điều chỉnh (Adjusted Odds Ratio) & 95% CI:* Ở Chó: `Senior` `aOR = 1.336 (1.285–1.388)`, `Junior` `aOR = 0.779 (0.723–0.840)`, cân nặng `+1 SD` `aOR = 1.072 (1.052–1.092)`, `3+ thuốc` `aOR = 1.774 (1.694–1.857)`. Ở Mèo: `Senior` `aOR = 1.365 (1.251–1.490)`, `Junior` `aOR = 1.132 (0.955–1.341, p = 0.154)`, cân nặng `+1 SD` `aOR = 0.909 (0.866–0.954)`, `3+ thuốc` `aOR = 2.876 (2.603–3.177)`. Giới tính (`Female vs Male`) không có ý nghĩa lâm sàng.
  - *Kết quả kiểm tra độ vững D6:* Toàn bộ chiều hướng và ý nghĩa thống kê giữ nguyên vững vàng qua các biến thể nội mẫu năm 2025 (`R1` cắt 1%–99%, `R2` giữ nhóm `'Unknown'` trên `66,544` ca, `R3` kiểm soát thêm `primary_reporter`), đồng thời các động lực chi phối chính (đặc biệt là đa trị liệu `3+ thuốc` và giai đoạn sống ở Chó) tiếp tục nhất quán khi đối chiếu ngoài mẫu trên `13,351` ca của `2026q1` (`R4`).


---
## 4. Câu hỏi Q3: Tỷ suất Đồng xuất hiện (ROR) Hoạt chất x Triệu chứng (Safety vs Efficacy)
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt
- **Notebook chi tiết:** [`02_q3_analysis.ipynb`](02_q3_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Cặp Hoạt chất — Triệu chứng có tín hiệu báo cáo mất cân đối mạnh nhất (ROR > 2):* Khi mở rộng kiểm định trên toàn bộ không gian cặp $(E, R)$ thỏa ngưỡng cỡ mẫu tối thiểu ($n_{\text{exp}} \ge 50, n_{\text{rea}} \ge 50, a \ge 20$) rồi mới hiệu chỉnh Benjamini–Hochberg FDR ($q_{\text{FDR}} < 0.05$), nhóm **Safety Issue** ($N = 37,811$ báo cáo khớp, $M = 3,372$ cặp giả thuyết được kiểm định) ghi nhận **$1,996$ cặp** đạt đồng thời $q_{\text{FDR}} < 0.05$ và $\text{CI}_{\text{low}} > 1.0$ (trong đó có **$1,566$ tín hiệu mạnh** $\text{ROR} \ge 2.0$). Đứng đầu bảng xếp hạng ROR là các tín hiệu đặc hiệu cơ quan đích của thuốc nội tiết – chuyển hóa và thuốc nhỏ tai: **`Bexagliflozin × Elevated betahydroxybutyrate (BHP)`** ($a = 93 / 996$ ca, **$9.34\%$**, $\text{ROR} = 3791.48$, 95% CI: $[527.89, 27231.39]$, $\text{PRR} = 3437.55$) cùng **`Velagliflozin × Ketosis`** ($a = 531 / 1,294$, **$41.04\%$**, $\text{ROR} = 68.93$, 95% CI: $[59.25, 80.19]$); **`Desoxycortone Pivalate × High sodium-potassium ratio (Na:K ratio)`** ($a = 44 / 98$, **$44.90\%$**, $\text{ROR} = 1279.56$, 95% CI: $[727.64, 2250.12]$); **`Trilostane × Abnormal ACTH stimulation test`** ($a = 52 / 519$, $\text{ROR} = 593.09$) và **`Hypocortisolaemia`** ($a = 64 / 519$, $\text{ROR} = 238.29$); cụm độc tính thính giác – tiền đình của chế phẩm nhỏ tai **`Hydrocortisone Aceponate × Loss of hearing`** ($a = 65$, $\text{ROR} = 387.27$), **`Miconazole Nitrate × Loss of hearing`** ($a = 68$, $\text{ROR} = 214.87$), **`Clotrimazole × Deafness`** ($a = 146$, $\text{ROR} = 163.31$), **`Mometasone Furoate × Deafness`** ($a = 222$, $\text{ROR} = 128.44$), **`Terbinafine × Head tilt - ear disorder`** ($a = 57$, $\text{ROR} = 116.95$); cùng các tín hiệu quy mô lớn như **`Bedinvetmab × Polydipsia`** ($a = 512$, $\text{ROR} = 7.95$) & **`Ataxia`** ($a = 632$, $\text{ROR} = 4.00$) và **`Oclacitinib × Overdose`** ($a = 276$, $\text{ROR} = 6.67$).
  - *Khác biệt cấu trúc giữa Biến cố bất lợi lâm sàng (Safety Issue) vs. Kém hiệu quả thuốc (Lack Of Expected Effectiveness — LOEE):* Trái với độc tính dược lý của Safety Issue, nhóm **LOEE** ($N = 23,705$, $M = 203$ cặp kiểm định, **$142$ cặp** đạt $q_{\text{FDR}} < 0.05$ & $\text{CI}_{\text{low}} > 1.0$, **$107$ cặp** có $\text{ROR} \ge 2.0$) phản ánh sự thất bại phòng trị hoặc kháng thuốc: đứng đầu là kháng nguyên vắc-xin Lyme tái tổ hợp (đã gộp mã thương mại USDA `Recombitek Lyme - 2126.R0` và `Recombinant E. Coli BL21/ECL19`) **`Borrelia Burgdorferi Ospa (Recombinant E. Coli) × Lack of efficacy (bacteria) - Borrelia`** ($a = 1,129 / 1,134$, **$99.56\%$**, $\text{ROR} = 3030.77$, 95% CI: $[1257.14, 7306.72]$), tiếp theo là hệ quả tử vong/mổ khám khi thất bại phòng trị hô hấp phức hợp (BRD) ở đàn bò (`Tulathromycin`, `Bovine Parainfluenza 3 Virus`, `Infectious Bovine Rhinotracheitis Virus`, `Bovine Respiratory Syncytial Virus` với `Death`/`Necropsy performed`, $\text{ROR} = 798.04 \text{–} 2754.47$), nhóm nhỏ tai kháng nấm (`Gentamicin`, `Posaconazole`, `Mometasone Furoate × LOEE (fungi) - NOS`, $\text{ROR} = 317.02 \text{–} 337.60$), và thuốc ký sinh trùng/mạn tính quy mô lớn (`Bedinvetmab × Partial lack of efficacy` $a = 214$, $\text{ROR} = 238.47$; `Fluralaner × LOEE - flea` $a = 2,624$, $\text{ROR} = 45.24$; `Ivermectin × LOEE - heartworm` $a = 1,826$, $\text{ROR} = 6.72$; `Sarolaner × LOEE - Borrelia` $a = 1,644$, $\text{ROR} = 4.77$). Khi tách riêng phân tích đường dùng (`route`, từ `64,095` báo cáo hợp lệ) theo hai nhóm: trong **Safety Issue** ($N = 35,724$, $M = 1,291$, $592$ cặp đạt chuẩn FDR), đường **`Topical`** gắn chặt với tổn thương tại chỗ bôi (`Application site hair loss` $a = 741$, $\text{ROR} = 741.86$; `Licking at application site` $\text{ROR} = 268.34$; `Application site alopecia` $\text{ROR} = 239.96$), đường **`Auricular (Otic)`** gắn với điếc và rối loạn tiền đình (`Deafness` $a = 266$, $\text{ROR} = 344.52$; `Loss of hearing` $\text{ROR} = 180.89$; `Head tilt` $\text{ROR} = 125.58$), còn đường **`Oral`** gắn với nhiễm toan ceton do SGLT2i (`Elevated BHP` $\text{ROR} = 102.40$; `Ketosis` $\text{ROR} = 43.26$); ngược lại trong **LOEE** ($N = 22,736$, $M = 64$, $31$ cặp đạt chuẩn FDR), đường **`Nasal`** và **`Subcutaneous`** gắn với thất bại vắc-xin/kháng sinh ở gia súc (`Nasal × Death` $\text{ROR} = 424.59$; `Subcutaneous × Necropsy performed` $\text{ROR} = 322.56$, `Partial lack of efficacy` $\text{ROR} = 19.24$), còn đường **`Oral`** và **`Topical`** gắn với kém hiệu quả phòng bệnh do ve và bọ chét (`Oral × LOEE - Ehrlichia` $\text{ROR} = 61.14$, `Anaplasma` $\text{ROR} = 23.64$, `Borrelia` $\text{ROR} = 17.14$; `Topical × LOEE - flea` $\text{ROR} = 10.15$).
  - *Khoảng tin cậy 95% CI, phân tầng loài (D4 & D2) và lưu ý diễn giải cảnh giác dược:* Phân tầng theo **Chó (`Dog`) vs. Mèo (`Cat`)** và tính $\text{ROR}_{\text{MH}}$ chuẩn xác (Cochran–Mantel–Haenszel kèm 95% CI Robins–Breslow–Greenland) giúp loại bỏ **$-78.1\%$** độ phóng đại giả tạo do gộp mẫu số ở thuốc chỉ định chuyên biệt cho Mèo như **`Velagliflozin × Ketosis`** (Crude $\text{ROR} = 66.98 \rightarrow \text{ROR}_{\text{MH}} = 14.64$, 95% CI: $[12.57, 17.06]$), hiệu chỉnh tăng cho **`Bedinvetmab × Polydipsia`** ($\text{ROR}_{\text{MH}} = 8.58$, 95% CI: $[7.34, 10.04]$, $+10.2\%$), và phát hiện nhiễu loài đảo chiều ở **`Selamectin × LOEE - flea`** (Crude $\text{ROR} = 4.44 \rightarrow \text{ROR}_{\text{MH}} = 1.77$, 95% CI: $[1.46, 2.15]$, giảm $-60.0\%$ xuống dưới ngưỡng tín hiệu $2.0$ do ở Mèo $\text{ROR}_{\text{Cat}} = 0.88$). Toàn bộ 6 tín hiệu chủ lực đều duy trì $\text{ROR} > 2.0$ ổn định qua 5 quý liên tiếp sang quý kiểm thử độc lập `2026 Q1` (`Borrelia OspA Vaccine × Borrelia LOEE` đạt $\text{ROR} = 1151.93$; `Velagliflozin × Ketosis` đạt $\text{ROR} = 289.20$; `Fluralaner × Flea LOEE` đạt $\text{ROR} = 148.80$; `Mometasone Furoate × Deafness` đạt $\text{ROR} = 114.09$; `Bedinvetmab × Polydipsia` đạt $\text{ROR} = 6.58$; và `Ivermectin × Heartworm LOEE` đạt $\text{ROR} = 6.21$). Vì dữ liệu openFDA CVM là hệ thống báo cáo tự nguyện, ROR cao chỉ phản ánh **tín hiệu báo cáo mất cân đối thống kê** (*strong disproportionality signal*) chứ không khẳng định quan hệ nhân quả trực tiếp hay tỷ lệ mắc bệnh thực tế.


---
## 5. Câu hỏi Q4: Sử dụng Thuốc Sai Nhãn (Off-label), Đa Trị Liệu & Kết cục Y khoa
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt
- **Notebook chi tiết:** [`02_q4_analysis.ipynb`](02_q4_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Tỷ lệ gia tăng nguy cơ tử vong khi dùng off-label:* (Sẽ được trích xuất từ `02_q4_analysis.ipynb` khi hoàn tất phân tích)
  - *Tác động cộng hưởng giữa Off-label và Đa trị liệu (Polypharmacy):* (Sẽ được trích xuất từ `02_q4_analysis.ipynb`)
  - *Kiểm soát biến nhiễu người báo cáo (`primary_reporter`) và kết quả D6:* (Sẽ được trích xuất từ `02_q4_analysis.ipynb`)


---
## 6. Câu hỏi Q5: Các yếu tố liên quan đến Tỷ lệ Động vật Bị ảnh hưởng ở Đàn (Herd Attack Rate)
- **Chủ sở hữu:** Trần Ngô Uyên Nhi
- **Notebook chi tiết:** [`02_q5_analysis.ipynb`](02_q5_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Mức độ ảnh hưởng trung vị theo loài và đường dùng:* Trên 964 báo cáo đàn hợp lệ, attack_rate trung vị tổng thể là 0.4325. Intramuscular và Topical có trung vị cao nhất (1.0000), Oral thấp nhất (0.1647), còn Cattle là 0.4000 và Chicken là 0.2083; Kruskal-Wallis cho đường dùng cho H = 68.7720, p = 1.8454e-13.
  - *Bootstrap 95% CI và quy mô đàn:* Oral có CI 95% trung vị là 0.1011–0.2500; Subcutaneous là 0.3000–0.4000; Topical là 1.0000–1.0000. Quy mô Nhỏ (<20 con) có trung vị 1.0000, Vừa (20–143 con) 0.2977 và Lớn (>143 con) 0.1348; kiểm định quy mô cho H = 286.7002, p = 5.5443e-63.
  - *Kiểm tra độ vững D6:* Chỉ có 1 ca attack_rate > 1 (1.2857). Cả hai kịch bản loại ca ngoại lai (R1) và chặn lên 1.0 (R2) cho kết quả trung vị theo đường dùng gần như không đổi, do đó kết luận chính ổn định nhưng vẫn bị giới hạn bởi mẫu báo cáo tự nguyện và kích thước nhóm loài không đều.
